# NSMC 초기 EDA

NSMC(Naver Sentiment Movie Corpus) 데이터셋의 기초 탐색.
- 클래스 분포
- 문서 길이 분포
- 결측값 현황

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_loader import load_nsmc

plt.rcParams['font.family'] = 'AppleGothic'  # macOS 한글 폰트
plt.rcParams['axes.unicode_minus'] = False

train_df, test_df = load_nsmc()
print(f'Train: {len(train_df):,}건, Test: {len(test_df):,}건')

## 기본 정보

In [ ]:
print('=== Train 기본 정보 ===')
print(train_df.info())
print()
print('=== Train 샘플 ===')
train_df.head()

## 클래스 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (df, title) in zip(axes, [(train_df, 'Train'), (test_df, 'Test')]):
    counts = df['label'].value_counts().sort_index()
    ax.bar(['부정(0)', '긍정(1)'], counts.values, color=['#e74c3c', '#2ecc71'])
    ax.set_title(f'{title} 클래스 분포')
    ax.set_ylabel('건수')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 100, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'artifacts' / 'eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('Train 클래스 분포:')
print(train_df['label'].value_counts(normalize=True).rename({0: '부정', 1: '긍정'}))

## 문서 길이 분포

In [ ]:
train_lengths = train_df['document'].str.len()
test_lengths = test_df['document'].str.len()

print('=== Train 문서 길이 통계 ===')
print(train_lengths.describe())
print()
print('=== Test 문서 길이 통계 ===')
print(test_lengths.describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (lengths, title) in zip(axes, [(train_lengths, 'Train'), (test_lengths, 'Test')]):
    ax.hist(lengths, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
    ax.axvline(lengths.mean(), color='red', linestyle='--', label=f'평균: {lengths.mean():.1f}')
    ax.axvline(lengths.median(), color='orange', linestyle='--', label=f'중앙값: {lengths.median():.1f}')
    ax.set_title(f'{title} 문서 길이 분포 (문자 단위)')
    ax.set_xlabel('문서 길이')
    ax.set_ylabel('빈도')
    ax.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'artifacts' / 'eda_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 결측값 현황

In [ ]:
print('=== 결측값 현황 (null 처리 후) ===')
print('Train:')
print(train_df.isnull().sum())
print()
print('Test:')
print(test_df.isnull().sum())
print()
print(f'Train document null 개수: {train_df["document"].isnull().sum()}')
print(f'Test document null 개수: {test_df["document"].isnull().sum()}')